<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Baseline Final - Random Forest (HOG + Slope)
**Maestría en Ingeniería - Universidad EAFIT** 
**Proyecto:** Detección de deslizamientos mediante IA

Este notebook establece el rendimiento base (baseline) y genera análisis avanzados de importancia de variables y métricas de error.

## 1. Configuración y Carga de Datos

In [ ]:
from google.colab import drive
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve

drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
detectar_img = list(base_path.glob('**/TrainData/img'))

if detectar_img:
    train_img_dir = detectar_img[0]
    train_mask_dir = detectar_img[0].parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ Éxito: {len(img_list)} muestras detectadas.")
else:
    print("❌ ERROR: Verifica la ruta en Drive.")

## 2. Extracción de Características

In [ ]:
N_SAMPLES = 1000 
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc="Procesando"):
    with h5py.File(img_list[i], "r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], "r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    
    # RGB y Normalización NumPy 2.0
    rgb = patch[:,:,[3,2,1]]
    rgb_norm = ((rgb - np.min(rgb)) / (np.ptp(rgb) + 1e-8) * 255).astype("uint8")
    
    # HOG + Slope (Banda 13)
    feats_hog = hog(rgb_norm, pixels_per_cell=(16,16), cells_per_block=(2,2), channel_axis=-1, feature_vector=True)
    avg_slope = np.mean(patch[:,:,12])
    
    X.append(np.hstack([feats_hog, avg_slope]))
    y.append(int(np.max(mask) > 0))

X, y = np.array(X), np.array(y)
print(f"✓ Dataset listo: {X.shape}")

## 3. Entrenamiento y Análisis de Resultados

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X[t_idx], y[t_idx])
    
    preds = rf.predict(X[v_idx])
    f1 = f1_score(y[v_idx], preds)
    f1_scores.append(f1)
    print(f"Fold {fold+1}: F1 = {f1:.4f}")

print(f"\n🚀 BASELINE FINAL: {np.mean(f1_scores):.4f}")

# --- GRÁFICAS DE VALOR PARA LA TESIS ---

# 1. Importancia de Variables
importances = rf.feature_importances_
indices = np.argsort(importances)[-15:]
plt.figure(figsize=(10, 5))
plt.title("Top 15 Características más influyentes")
plt.barh(range(len(indices)), importances[indices], color='skyblue')
plt.yticks(range(len(indices)), [f'HOG_{i}' if i < X.shape[1]-1 else 'SLOPE (B13)' for i in indices])
plt.show()

# 2. Matriz de Confusión
cm = confusion_matrix(y[v_idx], preds)
ConfusionMatrixDisplay(cm, display_labels=['No Desliz.', 'Desliz.']).plot(cmap='Blues')
plt.show()